# Phase 3. LTV (Lifetime Value) 코호트 분석

**프로젝트:** 카카오톡 선물하기 CRM 분석 포트폴리오  
**분석 기간:** 2023-01-01 ~ 2024-12-31  
**BigQuery:** `ds-ysy.kakao_gift`

## 분석 목표
1. 월별 코호트 정의 — 첫 구매월 기준으로 유저를 24개 그룹으로 분류
2. Retention Rate (재구매율) 히트맵 — 각 코호트가 M+1, M+2 ... M+12 시점에 몇 %가 남았는지
3. 누적 LTV (Lifetime Value — 고객 생애 가치) 커브 — 코호트 기준 시점별 1인당 평균 누적 구매금액
4. 코호트 품질 비교 — 시즌 코호트 vs 비시즌 코호트 LTV 차이
5. CRM 전략 시사점 도출

## Section 0. 환경 설정

In [ ]:
# Colab에서 BigQuery 인증
from google.colab import auth
auth.authenticate_user()  # 실행 시 팝업 → Google 계정으로 로그인

!pip install -q koreanize-matplotlib

from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import koreanize_matplotlib  # 한글 폰트
import os

PROJECT = 'ds-ysy'
DATASET = 'kakao_gift'
client  = bigquery.Client(project=PROJECT)

os.makedirs('charts', exist_ok=True)  # charts 폴더 없으면 자동 생성

def bq(sql: str) -> pd.DataFrame:
    """BigQuery 쿼리 실행 헬퍼"""
    return client.query(sql).to_dataframe()

---
## Section 1. 코호트 정의 — 유저별 첫 구매월 계산

**코호트(Cohort)란?**  
같은 달에 처음으로 구매한 유저 그룹.  
예: 2023-01 코호트 = 2023년 1월에 생애 첫 구매를 한 유저들.

왜 코호트로 나누나? → 유저마다 서비스를 시작한 시점이 다르기 때문에,  
단순 평균으로는 '초기 유저 vs 신규 유저'의 품질 차이를 볼 수 없음.

In [ ]:
# 유저별 첫 구매월(cohort_month) 계산
query_cohort = """
SELECT
    sender_user_id,
    DATE_TRUNC(MIN(created_at), MONTH) AS cohort_month
FROM `ds-ysy.kakao_gift.orders`
WHERE order_status = 'accepted'  -- completed 아님, 실제 완료 상태값
GROUP BY sender_user_id
"""
df_cohort = bq(query_cohort)
df_cohort['cohort_month'] = pd.to_datetime(df_cohort['cohort_month']).dt.to_period('M')  # YYYY-MM 형식으로 변환

print(f'전체 유저 수: {len(df_cohort):,}명')
print(f'코호트 수: {df_cohort["cohort_month"].nunique()}개')
df_cohort['cohort_month'].value_counts().sort_index()

---
## Section 2. 월별 구매 데이터 준비 — 코호트 기준 몇 번째 달인지 계산

In [ ]:
# 전체 주문 데이터 + cohort_month 조인
query_orders = """
SELECT
    o.sender_user_id,
    DATE_TRUNC(o.created_at, MONTH) AS order_month,
    o.total_amount_krw
FROM `ds-ysy.kakao_gift.orders` o
WHERE o.order_status = 'accepted'  -- completed 아님, 실제 완료 상태값
"""
df_orders = bq(query_orders)
df_orders['order_month'] = pd.to_datetime(df_orders['order_month']).dt.to_period('M')  # YYYY-MM 형식

# cohort_month 붙이기
df_orders = df_orders.merge(df_cohort, on='sender_user_id', how='left')

# 코호트 기준 몇 번째 달인지 계산 (months_since_first)
# 예: cohort_month=2023-01, order_month=2023-03 → months_since_first=2
df_orders['months_since_first'] = (df_orders['order_month'] - df_orders['cohort_month']).apply(lambda x: x.n)  # Period 뺄셈으로 경과 월 계산

print(df_orders.shape)
df_orders.head()

---
## Section 3. Retention Rate (재구매율) 히트맵

**Retention Rate란?**  
코호트 유저 중, 특정 시점에도 여전히 구매하고 있는 비율.  
예: 2023-01 코호트 100명 중 M+1(1개월 후)에 15명이 재구매 → Retention 15%

선물하기는 일반 앱과 달리 '로그인'이 아닌 '구매'로 활성을 정의함.

In [ ]:
# 코호트 × months_since_first 별 구매 유저 수 집계
df_retention = (
    df_orders
    .groupby(['cohort_month', 'months_since_first'])['sender_user_id']
    .nunique()  # 중복 제거 후 유저 수
    .reset_index()
    .rename(columns={'sender_user_id': 'active_users'})
)

# 코호트별 초기 유저 수(M+0) 추출
cohort_size = (
    df_retention[df_retention['months_since_first'] == 0]
    [['cohort_month', 'active_users']]
    .rename(columns={'active_users': 'cohort_size'})
)

# Retention Rate 계산 = active_users / cohort_size
df_retention = df_retention.merge(cohort_size, on='cohort_month', how='left')
df_retention['retention_rate'] = df_retention['active_users'] / df_retention['cohort_size']

# 피벗 테이블 — 행: 코호트월, 열: months_since_first
pivot_retention = df_retention.pivot_table(
    index='cohort_month',
    columns='months_since_first',
    values='retention_rate'
)
pivot_retention.index = pivot_retention.index.astype(str)  # Period → 'YYYY-MM' 문자열

pivot_retention

In [ ]:
# Retention 히트맵 시각화
# 카카오 브랜드 컬러 기반 커스텀 cmap: 흰색 → 카카오 옐로우(#FFE812) → 카카오 블랙(#000000)
from matplotlib.colors import LinearSegmentedColormap
kakao_cmap = LinearSegmentedColormap.from_list(
    'kakao', ['#FFFFFF', '#FFE812', '#000000']
)

fig, ax = plt.subplots(figsize=(16, 9))

sns.heatmap(
    pivot_retention,
    annot=True,           # 각 셀에 수치 표시
    fmt='.0%',            # 퍼센트 포맷
    cmap=kakao_cmap,      # 흰색→카카오 옐로우→블랙
    vmin=0, vmax=0.5,     # 색상 범위 고정 (0~50%)
    linewidths=0.5,
    ax=ax
)

ax.set_title('코호트별 Retention Rate (재구매율)', fontsize=14, pad=12)
ax.set_xlabel('첫 구매 이후 경과 월 (M+N)', fontsize=11)
ax.set_ylabel('코호트 (첫 구매월)', fontsize=11)
plt.tight_layout()
plt.savefig('charts/layer3_retention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4. 누적 LTV (Lifetime Value) 커브

**누적 LTV란?**  
코호트 기준 시점(M+0)부터 M+N까지, 1인당 평균 누적 구매금액.  
'이 유저들은 첫 구매 후 12개월 동안 평균 얼마를 썼는가?'

CAC (Customer Acquisition Cost — 고객 1명 획득에 드는 비용) 상한선을 설정할 때 핵심 지표.  
'12개월 LTV가 20만원이면, CAC는 그보다 낮아야 수익이 남는다.'

In [ ]:
# 코호트 × months_since_first 별 평균 누적 구매금액 계산
df_ltv = (
    df_orders
    .groupby(['cohort_month', 'months_since_first', 'sender_user_id'])['total_amount_krw']
    .sum()  # 유저별 해당 월 총 구매금액
    .reset_index()
)

# 코호트 × months_since_first 별 누적 합산 → 1인당 평균
df_ltv_agg = (
    df_ltv
    .groupby(['cohort_month', 'months_since_first'])['total_amount_krw']
    .sum()
    .reset_index()
    .merge(cohort_size, on='cohort_month', how='left')
)
df_ltv_agg['avg_revenue_per_user'] = df_ltv_agg['total_amount_krw'] / df_ltv_agg['cohort_size']

# 누적합 계산 (cumsum) — 각 시점의 값은 M+0부터 해당 시점까지의 합
df_ltv_agg = df_ltv_agg.sort_values(['cohort_month', 'months_since_first'])
df_ltv_agg['cumulative_ltv'] = (
    df_ltv_agg.groupby('cohort_month')['avg_revenue_per_user'].cumsum()
)

df_ltv_agg.head(10)

In [ ]:
# 전체 코호트 평균 누적 LTV 커브
avg_ltv_curve = (
    df_ltv_agg
    .groupby('months_since_first')['cumulative_ltv']
    .mean()  # 24개 코호트 평균
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    avg_ltv_curve.index,
    avg_ltv_curve.values,
    marker='o', linewidth=2.5, color='#000000'  # 카카오 블랙
)
ax.fill_between(
    avg_ltv_curve.index,
    avg_ltv_curve.values,
    alpha=0.25, color='#FFE812'  # 카카오 옐로우로 면적 강조
)

# 주요 시점 수치 표시
for m in [0, 1, 2, 5, 11]:
    if m in avg_ltv_curve.index:
        val = avg_ltv_curve[m]
        ax.annotate(
            f'M+{m}\n{val:,.0f}원',
            xy=(m, val),
            xytext=(m, val + 3000),
            fontsize=8, ha='center', color='#000000'
        )

ax.set_title('전체 코호트 평균 누적 LTV 커브', fontsize=13)
ax.set_xlabel('첫 구매 이후 경과 월 (M+N)', fontsize=11)
ax.set_ylabel('1인당 누적 구매금액 (원)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/layer3_ltv_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5-A. 코호트별 M+0 vs M+5 LTV 막대 비교

LTV 커브가 전체 평균이었다면, 이 차트는 **코호트별 개별 성장폭**을 보여줘요.  
M+0(첫 달)과 M+5(6개월 누적)를 나란히 두면 어느 코호트가 장기 구매로 이어졌는지 한눈에 비교 가능.

In [ ]:
# 코호트별 M+0 vs M+5 누적 LTV 막대 비교
pivot_ltv_bar = df_ltv_agg.pivot_table(
    index='cohort_month',
    columns='months_since_first',
    values='cumulative_ltv'
)

cohorts = pivot_ltv_bar.index.astype(str)  # Period → 'YYYY-MM' 문자열
ltv_m0 = pivot_ltv_bar[0] if 0 in pivot_ltv_bar.columns else None  # 첫 달 LTV
ltv_m5 = pivot_ltv_bar[5] if 5 in pivot_ltv_bar.columns else None  # 6개월 누적 LTV

x = np.arange(len(cohorts))  # x축 위치
w = 0.35                      # 막대 너비

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - w/2, ltv_m0.values, width=w, label='M+0 (첫 달)',      color='#FFE812', linewidth=0)  # 카카오 옐로우, outline 없음
ax.bar(x + w/2, ltv_m5.values, width=w, label='M+5 (6개월 누적)', color='#000000', linewidth=0, alpha=0.85)  # 카카오 블랙, outline 없음

ax.set_xticks(x)
ax.set_xticklabels(cohorts, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('1인당 누적 구매금액 (원)', fontsize=10)
ax.set_title('코호트별 LTV 비교 (M+0 vs M+5 누적)', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/layer3_cohort_bar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5-B. 코호트 품질 비교 — 시즌 vs 비시즌

선물하기의 특성: 빼빼로데이(11월), 설날(1~2월), 어버이날(5월) 등 시즌에 신규 유저가 대거 유입됨.  
이 유저들은 이벤트성으로 한 번만 구매하고 이탈할까, 아니면 남을까?

**시즌 코호트:** 1월, 5월, 11월  
**비시즌 코호트:** 6월, 7월, 8월 (선물 이벤트 부재 기간)

In [ ]:
# 시즌/비시즌 코호트 분류
season_months    = [1, 5, 11]  # 설날, 어버이날, 빼빼로데이 시즌
offseason_months = [6, 7, 8]   # 비시즌

df_ltv_agg['cohort_month_dt'] = df_ltv_agg['cohort_month'].dt.to_timestamp()  # Period → timestamp 변환
df_ltv_agg['season_type'] = df_ltv_agg['cohort_month_dt'].dt.month.map(
    lambda m: '시즌' if m in season_months else ('비시즌' if m in offseason_months else None)
)

# 시즌 타입별 평균 누적 LTV 커브
df_season_ltv = (
    df_ltv_agg[df_ltv_agg['season_type'].notna()]
    .groupby(['season_type', 'months_since_first'])['cumulative_ltv']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))

colors = {'시즌': '#FFE812', '비시즌': '#AAAAAA'}  # 카카오 옐로우 vs 회색
for stype, grp in df_season_ltv.groupby('season_type'):
    ax.plot(
        grp['months_since_first'],
        grp['cumulative_ltv'],
        marker='o', linewidth=2.5,
        color=colors[stype], label=stype,
        markeredgecolor='#000000', markeredgewidth=0.8  # 마커 테두리 블랙
    )

ax.set_title('시즌 vs 비시즌 코호트 — 누적 LTV 비교', fontsize=13)
ax.set_xlabel('첫 구매 이후 경과 월 (M+N)', fontsize=11)
ax.set_ylabel('1인당 누적 구매금액 (원)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/layer3_cohort_ltv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6. 코호트별 LTV 피벗 테이블 (상세 수치)

In [ ]:
# 코호트별 주요 시점 누적 LTV 피벗
pivot_ltv = df_ltv_agg.pivot_table(
    index='cohort_month',
    columns='months_since_first',
    values='cumulative_ltv'
)
pivot_ltv.index = pivot_ltv.index.astype(str)  # Period → 'YYYY-MM' 문자열

# 주요 시점만 선택
cols_to_show = [c for c in [0, 1, 2, 5, 8, 11] if c in pivot_ltv.columns]
subset = pivot_ltv[cols_to_show].copy()

# 컬럼명 M+N 형식으로 변경
subset.columns = [f'M+{c}' for c in subset.columns]

# ── 색상 계산 (YlOrRd 기반) ─────────────────────────────
cmap = plt.get_cmap('YlOrRd')
vmin = subset.min().min()
vmax = subset.max().max()

def get_bg(val):
    if pd.isna(val):
        return (1, 1, 1, 1)  # NaN → 흰색
    return cmap((val - vmin) / (vmax - vmin))

# ── matplotlib 테이블 렌더링 ────────────────────────────
n_rows, n_cols = subset.shape
fig, ax = plt.subplots(figsize=(n_cols * 1.6 + 1.5, n_rows * 0.42))  # 높이 줄임
ax.axis('off')

col_labels = ['코호트'] + subset.columns.tolist()
cell_text, cell_colors = [], []

for idx, row in subset.iterrows():
    text_row  = [idx]
    color_row = [(0.95, 0.95, 0.95, 1)]
    for val in row:
        text_row.append(f'{val:,.0f}원' if not pd.isna(val) else '')
        color_row.append(get_bg(val))
    cell_text.append(text_row)
    cell_colors.append(color_row)

table = ax.table(
    cellText=cell_text,
    cellColours=cell_colors,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
    bbox=[0, 0, 1, 1]  # 테이블이 ax 전체를 꽉 채우도록
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)

# 헤더 스타일
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#000000')
    table[0, j].set_text_props(color='white', fontweight='bold')

# 모든 셀 테두리 제거
for cell in table.get_celld().values():
    cell.set_linewidth(0)

plt.suptitle('코호트별 누적 LTV (M+N 시점별)', fontsize=12, y=1.01)  # 제목을 표 바로 위에
plt.savefig('charts/layer3_ltv_pivot.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: charts/layer3_ltv_pivot.png')

---
## 인사이트 요약

In [ ]:
# 핵심 수치 출력
print('=== Layer 3. LTV 코호트 분석 핵심 수치 ===')
print()

for m in [0, 1, 2, 5, 11]:
    if m in avg_ltv_curve.index:
        print(f'M+{m:2d} 평균 누적 LTV: {avg_ltv_curve[m]:>10,.0f}원')

print()
m1_retention = pivot_retention[1].mean() if 1 in pivot_retention.columns else None  # 1개월 Retention 평균
if m1_retention:
    print(f'1개월 평균 Retention: {m1_retention:.1%}')

if 0 in avg_ltv_curve.index and 11 in avg_ltv_curve.index:
    multiple = avg_ltv_curve[11] / avg_ltv_curve[0]  # 12개월 LTV 성장 배율
    print(f'M+0 대비 M+11 LTV 성장 배율: {multiple:.1f}x')